# Feature Engineering for Cricket Analytics

This notebook creates features for machine learning models.

## Objectives:
- Create player-specific features
- Engineer team performance metrics
- Build contextual features (venue, conditions, etc.)
- Generate rolling statistics and momentum indicators

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent / 'src'))

from etl.data_loader import CricketDataLoader
from etl.data_processor import DataProcessor

print("Libraries imported successfully!")

## 1. Load Processed Data

In [ ]:
# Load data
loader = CricketDataLoader()
processor = DataProcessor()

# TODO: Load from CricPy API
match_data = pd.DataFrame()

print(f"Loaded {len(match_data)} matches")

## 2. Player Performance Features

In [ ]:
def create_player_features(df):
    """
    Create player-specific performance features.
    """
    if df.empty:
        return df
    
    features = df.copy()
    
    # Batting features
    if 'runs' in features.columns and 'balls_faced' in features.columns:
        features['strike_rate'] = (features['runs'] / features['balls_faced']) * 100
        features['boundary_percentage'] = (features['fours'] + features['sixes']) / features['balls_faced'] * 100
    
    # Bowling features
    if 'wickets' in features.columns and 'overs_bowled' in features.columns:
        features['bowling_average'] = features['runs_conceded'] / features['wickets'].replace(0, np.nan)
        features['economy_rate'] = features['runs_conceded'] / features['overs_bowled']
    
    # Consistency metrics
    if 'player' in features.columns:
        features['runs_std'] = features.groupby('player')['runs'].transform('std')
        features['avg_runs'] = features.groupby('player')['runs'].transform('mean')
    
    return features

# Apply feature engineering
match_data = create_player_features(match_data)
print("Player features created!")

## 3. Team Performance Features

In [ ]:
def create_team_features(df):
    """
    Create team-level performance features.
    """
    if df.empty or 'team' not in df.columns:
        return df
    
    features = df.copy()
    
    # Team batting strength
    if 'total_runs' in features.columns:
        features['team_avg_runs'] = features.groupby('team')['total_runs'].transform('mean')
        features['team_max_runs'] = features.groupby('team')['total_runs'].transform('max')
    
    # Team win rate
    if 'result' in features.columns:
        features['team_win_rate'] = features.groupby('team')['result'].transform(
            lambda x: (x == 'won').sum() / len(x)
        )
    
    # Head-to-head records
    if 'opponent' in features.columns:
        features['h2h_wins'] = features.groupby(['team', 'opponent'])['result'].transform(
            lambda x: (x == 'won').sum()
        )
    
    return features

match_data = create_team_features(match_data)
print("Team features created!")

## 4. Rolling Statistics (Form/Momentum)

In [ ]:
def create_rolling_features(df, window=5):
    """
    Create rolling window features to capture recent form.
    """
    if df.empty or 'date' not in df.columns:
        return df
    
    features = df.copy()
    features = features.sort_values(['team', 'date'])
    
    # Rolling averages for team performance
    if 'total_runs' in features.columns:
        features[f'rolling_avg_runs_{window}'] = features.groupby('team')['total_runs'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    # Rolling win rate
    if 'result' in features.columns:
        features['win_binary'] = (features['result'] == 'won').astype(int)
        features[f'rolling_win_rate_{window}'] = features.groupby('team')['win_binary'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    # Recent form (last 3 matches)
    features['recent_form'] = features.groupby('team')['win_binary'].transform(
        lambda x: x.rolling(3, min_periods=1).sum()
    )
    
    return features

match_data = create_rolling_features(match_data, window=5)
print("Rolling features created!")

## 5. Venue-specific Features

In [ ]:
def create_venue_features(df):
    """
    Create venue-specific features.
    """
    if df.empty or 'venue' not in df.columns:
        return df
    
    features = df.copy()
    
    # Venue scoring rate
    if 'total_runs' in features.columns:
        features['venue_avg_score'] = features.groupby('venue')['total_runs'].transform('mean')
        features['venue_max_score'] = features.groupby('venue')['total_runs'].transform('max')
    
    # Team performance at venue
    if 'team' in features.columns:
        features['team_venue_avg'] = features.groupby(['team', 'venue'])['total_runs'].transform('mean')
        features['team_venue_wins'] = features.groupby(['team', 'venue'])['result'].transform(
            lambda x: (x == 'won').sum() if 'result' in features.columns else 0
        )
    
    # Home advantage
    if 'home_team' in features.columns:
        features['is_home'] = (features['team'] == features['home_team']).astype(int)
    
    return features

match_data = create_venue_features(match_data)
print("Venue features created!")

## 6. Match Context Features

In [ ]:
def create_context_features(df):
    """
    Create contextual features (time, phase of tournament, etc.).
    """
    if df.empty:
        return df
    
    features = df.copy()
    
    # Temporal features
    if 'date' in features.columns:
        features['date'] = pd.to_datetime(features['date'])
        features['year'] = features['date'].dt.year
        features['month'] = features['date'].dt.month
        features['day_of_week'] = features['date'].dt.dayofweek
        features['is_weekend'] = features['day_of_week'].isin([5, 6]).astype(int)
    
    # Match importance (playoff/final stages)
    if 'match_type' in features.columns:
        features['is_knockout'] = features['match_type'].isin(['semi-final', 'final', 'qualifier']).astype(int)
    
    # Toss impact
    if 'toss_winner' in features.columns and 'match_winner' in features.columns:
        features['toss_match_winner'] = (features['toss_winner'] == features['match_winner']).astype(int)
    
    return features

match_data = create_context_features(match_data)
print("Context features created!")

## 7. Feature Selection & Importance

In [ ]:
if not match_data.empty:
    # List all engineered features
    all_features = match_data.columns.tolist()
    print(f"\nTotal features created: {len(all_features)}")
    print("\nFeature list:")
    for i, feat in enumerate(all_features, 1):
        print(f"{i}. {feat}")
else:
    print("Awaiting data integration to display features")

## 8. Feature Correlation Analysis

In [ ]:
if not match_data.empty:
    # Select numeric features
    numeric_features = match_data.select_dtypes(include=[np.number])
    
    if not numeric_features.empty:
        # Correlation with target (if exists)
        if 'result' in match_data.columns:
            target = (match_data['result'] == 'won').astype(int)
            correlations = numeric_features.corrwith(target).sort_values(ascending=False)
            
            print("\nTop 10 Features Correlated with Match Outcome:")
            print(correlations.head(10))
            
            # Visualization
            plt.figure(figsize=(10, 6))
            correlations.head(15).plot(kind='barh')
            plt.title('Top 15 Features by Correlation with Match Outcome')
            plt.xlabel('Correlation Coefficient')
            plt.tight_layout()
            plt.show()

## 9. Save Engineered Features

In [ ]:
# Save processed data
if not match_data.empty:
    output_path = Path.cwd().parent / 'data' / 'processed' / 'features.csv'
    match_data.to_csv(output_path, index=False)
    print(f"\nFeatures saved to: {output_path}")
    print(f"Shape: {match_data.shape}")
else:
    print("\nNo data to save - integrate CricPy API first")

## Next Steps

1. **Model Training** - Use these features in notebook 03 for match prediction
2. **Feature Selection** - Apply techniques like RFE, feature importance
3. **Advanced Features** - Create interaction terms, polynomial features
4. **Validation** - Check for data leakage and temporal consistency